# Ylivertainen v2 — Clinical Association Pipeline

End-to-end workflow for finding statistically valid associations between **target outcomes**
and **predictor variables** in a clinical dataset.

This notebook drives six universal modules:

| Module                       | Purpose                                                    |
|------------------------------|------------------------------------------------------------|
| `schema_infer.py`            | Auto-classify each column (continuous, ordinal, …)         |
| `cleaning.py`                | Apply the schema, audit duplicates, derive new columns     |
| `dda.py`                     | Per-column descriptive stats + SVG plots                   |
| `missingness_resolution.py`  | Missing pattern analysis, flags, MICE multiple imputation  |
| `eda.py`                     | Univariate target × predictor screening (FDR-corrected)    |
| `inferential.py`             | Multivariable logistic regression with Rubin pooling       |

**Pipeline order**

```
load → infer schema → clean → DDA → missingness → derive new cols → DDA again
   → EDA screen → MICE impute → multivariable logistic (Rubin pool) → outputs
```

All outputs land under `output/<stage>/{figures,tables}/` as SVG and CSV.


## 0. Setup

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
import pandas as pd
pd.set_option("display.max_columns", None)

import numpy as np
import shutil
from pathlib import Path

from schema_infer import (infer_schema, print_schema_template, print_column_uniques, schema_summary,
                          export_schema_summary, ColSpec)
from cleaning import (apply_schema, audit_duplicates, export_cleaning_artifacts,
                      write_cleaned_csv, bin_numeric, bin_datetime, make_missing_flag,
                      combine_categories)
from dda import run_dda
from missingness_resolution import (analyze_missingness, add_missing_flags,
                                    mark_structural_missing, drop_rows,
                                    mice_impute, simple_impute, imputation_audit)
from eda import screen_associations
from diagnostic_accuracy import screen_diagnostic_accuracy
from inferential import run_inferential, summarize_multivariable_cases

OUTPUT_ROOT = Path("output")
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from config import load

#🟧🟧🟧 None = all years; e.g. [2025] for one cohort year

ANALYSIS_YEARS: list[int] | None = None

# Run top to bottom

Each section: **edit → run → next**. No jumping back to a config block at the top.


## 01. Load data


In [ ]:
DATA_PATH = "Meningiomas PSKUS grants - Visi pacienti.csv"   # or "yourdata.csv"

_c01 = load("01_cohort")
df_raw = _c01.load_raw(DATA_PATH)
df_raw.head(0)


## 02. Column rename

1. Run **see raw columns** → copy the printed skeleton  
2. Paste into `COLUMN_RENAME_MAP` below and fill in snake_case names  
3. Run **apply rename**


In [ ]:
#🟧🟧🟧 Step 1 — see raw columns (run once per new dataset)

load("02_column_rename_map").list_cols(df_raw)


In [ ]:
#🟧🟧🟧 Step 2 — paste skeleton here and fill in the right-hand names

COLUMN_RENAME_MAP = {
    "Nr.": "id",
    "Personas kods": "patient_code",
    "Unnamed: 2": "entry_year",
    "Vecums. gadi": "age",
    "Dzimums. 0 - vīrietis\n1 - sieviete\"": "sex",
    "Histoloģija. 0 - nav\n1 - ir": "histology_available",
    "WHO pakāpe (2021). 1 / 2 / 3": "who_grade",
    "Progesterons. 0 - negatīvs\n1 - pozitīvs": "progesterone_pos",
    "Ki-67 (%). skaitlis. %": "ki67_pct",
    "Smadzeņu parenhīmas invāzija. 0 - nav\n1 - ir": "brain_invasion",

    "Nekroze histoloģiski. 0 - nav\n1 - ir": "hist_necrosis",
    "MRI izmeklējuma datums": "mri_date",
    "Puse. 1 - labā\n2 - kreisā\n3 - viduslīnija": "side",
    "Lokalizācija: skull base / non–skull base. 0 - non-skull base\n1 - skull base": "tumor_location",
    "Cik meningiomas?": "meningioma_count",

    "Max diametrs. skaitlis.cm": "max_diameter_cm",
    "Tilpums": "tumor_volume",
    "Pamatmodalitāte analīzei. 0 - MRI\n1 - CT\n3 - MRI+CT": "base_modality",
    "K/v i/v. 0 - nav\n1 - ir": "iv_contrast",
    "0 - primārs\n1 - recidīvs": "tumor_episode",
    "Audzēja robeža. 1 = gluda. \n2 = neregulāra": "tumor_margin",
    "Dural tail sign. 0 - nav\n1 - ir": "dural_tail",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement). 0 - nav\n1 - ir": "capsular_enhancement",
    "Kontrastēšanās veids. 0 - homogēna\n1 - heterogēna": "heterogeneous_enhancement",
    "Perifokāla tūska. 0 - nav\n1 - ir": "perifocal_edema",

    "Perifokālas tūskas tilpums. cm3": "edema_volume_cm3",
    "Masas efekts. 0 - nav\n1 - ir": "mass_effect",
    "Audzēja kalcifikācija. 0 - nav\n1 - ir": "calcification",
    "Cistiskas komponentes. 0 - nav\n1 - ir": "cystic_component",
    "Audzēja nekroze. 0 - nav\n1 - ir": "necrosis",
    "Hemorāģiskas sastāvdaļas. 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams ": "hemorrhage",
    "Kaule hiperostoze. 0 - nav\n1 - ir": "hyperostosis",
    "Kaula invāzija (cortical destruction). 0 - nav\n1 - ir": "cortical_destruction",
    "Tumor Hyperintensity on DWI. 0 - nav\n1 - ir": "dwi_hyperintensity",
    "Tumor Hyperintensity on T2. 0 - nav\n1 - ir": "t2_hyperintensity",
    "Tumor Hypointensity on T1. 0 - nav\n1 - ir": "t1_hypointensity",

    "Sīnuss. 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug": "sinus_invasion",
    "Cauraug falx cerebri 0 - nav. 1 - ir": "transfalcine_extension",
    "ADC map value": "adc_value",
    }

In [ ]:
#🟧🟧🟧 Step 3 — apply rename

df_raw = load("02_column_rename_map").apply_rename(df_raw, COLUMN_RENAME_MAP)
df = df_raw
df.head(0)

## 02b. Key columns & cohort filter

Use **renamed** names from §02. Edit, then run.


In [ ]:
YEAR_COLUMN = "entry_year"
ID_COLS = ["id", "patient_code", "entry_year"]

ANALYSIS_YEARS: list[int] | None = None   # e.g. [2025]; None = all years
# ANALYSIS_YEARS = 

In [ ]:
df_raw = _c01.filter_cohort(df_raw, YEAR_COLUMN, ANALYSIS_YEARS)
df = df_raw

## 03. Schema

1. Run **infer**  
2. Run **print template**  
3. Run **column uniques** (nulls / replace hints)  
4. Edit **schema_overrides**  
5. Run **apply overrides**


In [ ]:
schema = infer_schema(df_raw)
schema_summary(schema)


In [ ]:
print_schema_template(schema)


In [ ]:
#🟧🟧🟧 inspect raw values — use for nulls=() and replace={} below
print_column_uniques(df_raw, schema)


In [ ]:
#🟧🟧🟧 Edit overrides, then run

schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind="id", keep=False),
    
    'entry_year': ColSpec(name='entry_year', kind='datetime', keep=False, datetime_bin='year'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='nominal', replace={0:"male", 1:"female",}),
    
    'who_grade': ColSpec(name='who_grade', kind='ordinal', ordered_levels=["1","2","3"]),
    
    'histology_available': ColSpec(name='histology_available', kind='binary', nulls=(2,), keep=False),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary', nulls=(2,)),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    
    'mri_date': ColSpec(name='mri_date', kind='datetime', keep=False, datetime_bin='full'),
    
    'side': ColSpec(name='side', kind='nominal', replace={'1': "right", '2': "left", '3': "midline"}),
    'tumor_location': ColSpec(name='tumor_location', kind='nominal', replace={0: "non_skull_base", 1: "skull_base"}, nulls=(2,)),
    'meningioma_count': ColSpec(name='meningioma_count', kind='count'),
    'max_diameter_cm': ColSpec(name='max_diameter_cm', kind='continuous'),
    'tumor_volume': ColSpec(name='tumor_volume', kind='continuous'),
    
    'base_modality': ColSpec(name='base_modality', kind='nominal', replace={0: "mri", 1: "ct", 3: "mri_ct"}, keep=False),
    'iv_contrast': ColSpec(name='iv_contrast', kind='binary', keep=False),
    
    'tumor_episode': ColSpec(name='tumor_episode', kind='ordinal', replace={'0': "primary", '1': "recurrent"}, ordered_levels=["primary", "recurrent"], nulls=("multiplas",)),
    
    'tumor_margin': ColSpec(name='tumor_margin', kind='nominal', replace={1: "regular", 2: "irregular"}, nulls=(0,)),
    'dural_tail': ColSpec(name='dural_tail', kind='binary'),
    
    'capsular_enhancement': ColSpec(name='capsular_enhancement', kind='binary'),
    'heterogeneous_enhancement': ColSpec(name='heterogeneous_enhancement', kind='binary'),
    'dwi_hyperintensity': ColSpec(name='dwi_hyperintensity', kind='binary', nulls=('-',)),
    't2_hyperintensity': ColSpec(name='t2_hyperintensity', kind='binary', nulls=('-',)),
    't1_hypointensity': ColSpec(name='t1_hypointensity', kind='binary', nulls=('-',)),
    
    'perifocal_edema': ColSpec(name='perifocal_edema', kind='binary'),
    'edema_volume_cm3': ColSpec(name='edema_volume_cm3', kind='continuous'),
    
    'mass_effect': ColSpec(name='mass_effect', kind='binary'),
    'calcification': ColSpec(name='calcification', kind='binary'),
    'cystic_component': ColSpec(name='cystic_component', kind='binary'),
    'mri_necrosis': ColSpec(name='mri_necrosis', kind='binary'),
    'hemorrhage': ColSpec(name='hemorrhage', kind='binary', nulls=(2.0,)),
    'hyperostosis': ColSpec(name='hyperostosis', kind='binary'),
    'sinus_invasion': ColSpec(name='sinus_invasion', kind='ordinal', replace={0: "no_invasion", 1: "sinus_invasion", 2: "transsinus_extension"}, ordered_levels=["no_invasion", "sinus_invasion", "transsinus_extension"]),
    'cortical_destruction': ColSpec(name='cortical_destruction', kind='binary'),
    'transfalcine_extension': ColSpec(name='transfalcine_extension', kind='binary'),
    
    'adc_value': ColSpec(name='adc_value', kind='continuous'),
    }


In [ ]:
load("03_schema_overrides").apply_schema_overrides(schema, schema_overrides, OUTPUT_ROOT)

## 03b. Pre-schema inclusion filters

Apply on **raw strings** before `apply_schema` so excluded rows never become categorical levels (e.g. spinal meningioma in `side` / `mri_date`).

In [ ]:
_c04 = load("04_row_filters")

pre_schema_row_filters = [
    _c04.brain_meningioma_row_filter(),
]

df, pre_schema_row_filter_log = _c04.apply_row_filters(
    df, pre_schema_row_filters,
)
n_rows_pre_schema = len(df)
pre_schema_row_filter_log

## 04. Apply schema


In [ ]:
schema_log = []
df = apply_schema(df, schema, log=schema_log)
n_rows_after_schema = len(df)
df.head()

## 05. Duplicate audit


In [ ]:
dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
dupes.head() if len(dupes) else print('No duplicate groups found.')

## 06. Row filters (post-schema)

Edit `post_schema_row_filters` (toggle `active=True/False`), run the cell, then run finalize below.

Pre-schema inclusion filters (§03b) run on raw strings; post-schema filters run after datetime coercion etc.


In [ ]:
post_schema_row_filters = [
    _c04.RowFilter(
        name="who_grade exists",
        keep=lambda d: d["who_grade"].notna(),
        note="inclusion criteria - histological WHO grade",
        active=True,
    ),
    _c04.RowFilter(
        name="MRI exists",
        keep=lambda d: d["mri_date"].notna(),
        note="inclusion criteria - MRI",
        active=True,
    ),
    # _c04.RowFilter(
    #     name="sex known",
    #     keep=lambda d: d["sex"] != "unknown",
    #     note="Keep rows where sex is known (not 'unknown')",
    #     active=False,
    # ),
    # _c04.RowFilter(
    #     name="adult patients only",
    #     keep=lambda d: d["age"] >= 18,
    #     note="Keep rows where age is 18 or older",
    #     active=False,
    # ),
    # _c04.RowFilter(
    #     name="exclude WHO grade 2 or 3",
    #     keep=lambda d: ~d["who_grade"].isin(["2", "3"]),
    #     note="Keep rows where WHO grade is 1 (exclude grades 2 and 3)",
    #     active=False,
    # ),
]

df, post_schema_row_filter_log = _c04.apply_row_filters(df, post_schema_row_filters)
row_filter_log = _c04.combine_row_filter_logs(
    pre_schema_row_filter_log,
    post_schema_row_filter_log,
)
row_filter_log


In [ ]:
df = _c04.finalize_row_drops(
    df, row_filter_log,
    output_root=OUTPUT_ROOT,
    df_raw=df_raw,
    n_rows_pre_schema=n_rows_pre_schema,
    n_rows_after_schema=n_rows_after_schema,
    schema=schema,
    dupes=dupes,
    schema_log=schema_log,
)


## 07. DDA — first pass


In [ ]:
from cleaning import format_table_for_display
from IPython.display import display

dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))


## 08. Missingness analysis


In [ ]:
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary[missing_summary.n_missing > 0]

### 08a. Missingness policy

Declare structural and MNAR decisions in lists below (same pattern as row filters).
Run the next cell to apply them once and get an audit log.

- **Structural** — NaN means the slot does not exist (do not impute; derive count/max instead).
- **MNAR** — missingness itself may be informative (adds `<col>_missing` flag).


In [ ]:
_c05 = load("05_missingness")

STRUCTURAL_GROUPS = [
    # Example only. Keep empty if we do not currently have slot-style columns.
    # _c05.StructuralGroup(
    #     name="lesion_mri_pirads",
    #     cols=["lesion_1_MRI_PIRADS", "lesion_2_MRI_PIRADS", "lesion_3_MRI_PIRADS"],
    #     derive_count_col="n_mri_pirads_lesions",
    #     derive_max_col="max_mri_pirads",
    #     skip_raw=True,
    #     reason="Blank lesion slots mean lesion does not exist, not unknown.",
    # ),
]

MNAR_COLUMNS = [
    # Add only when missingness itself may be informative.
    # _c05.MnarColumn(
    #     col="ki67_pct",
    #     flag_col="ki67_pct_missing",
    #     reason="Ki-67 may be absent because it was not measured/reported in selected cases.",
    # ),
    # _c05.MnarColumn(
    #     col="adc_value",
    #     flag_col="adc_value_missing",
    #     reason="ADC may be absent when DWI/ADC was unavailable or non-diagnostic.",
    # ),
]


In [ ]:
df, schema, missingness_log = _c05.apply_missingness_policy(
    df=df,
    schema=schema,
    structural_groups=STRUCTURAL_GROUPS,
    mnar_columns=MNAR_COLUMNS,
)

missingness_log

## 09. Derivations

Declare derived columns in a list below (same pattern as row filters / missingness).
All study-specific logic lives in the notebook; `06_derivations.py` is just the engine.

**Building blocks:**

- **`BinNumeric`** — cut a numeric column into ordered bins (`age` → `age_bins`).
  Bins = edge values; labels = one per gap. `len(bins) - 1 == len(labels)`.
  Default `right=False`: left-closed intervals (`[50, 60)` → `"50-59"`).
- **`Apply`** — custom logic via helper + `fn=lambda s: ...` (Ki-67 midpoint, grouped labels, boolean flags, etc.).

Each entry supports `active=False` (skip) and `overwrite=True` (replace existing column).
Append to `DERIVATIONS` to add columns — no `.py` edits needed.


In [ ]:
_c06 = load("06_derivations")


def _ki67_midpoint(x):
    if pd.isna(x):
        return pd.NA
    parts = str(x).replace(",", ".").split("-")
    nums = [float(p) for p in parts]
    return sum(nums) / len(nums)
def _ki67_group(x):
    if pd.isna(x):
        return pd.NA
    if x <= 4:
        return "low_le_4"
    if x < 10:
        return "intermediate_5_9"
    return "high_ge_10"

DERIVATIONS = [
    _c06.BinNumeric(
        name="age_bins",
        source="age",
        bins=[-np.inf, 50, 60, 70, 80, np.inf],
        labels=["<50", "50-59", "60-69", "70-79", "80+"],
        kind="ordinal",
        active=True,
        overwrite=False,
        reason="Age groups for descriptive tables.",
    ),
    _c06.Apply(
        name="high_grade",
        source="who_grade",
        fn=lambda s: s.astype("Float64").pipe(lambda sf: (sf == 2) | (sf == 3)),
        kind="binary",
        active=True,
        overwrite=False,
        reason="WHO grade 2/3 = high-grade meningioma.",
    ),
    _c06.Apply(
        name="multiple_meningiomas",
        source="meningioma_count",
        fn=lambda s: s.astype("Float64") > 1,
        kind="binary",
        active=True,
        overwrite=False,
        reason=">1 meningioma = multiple",
    ),
    _c06.Apply(
        name="ki67_mid",
        source="ki67_pct",
        fn=lambda s: s.map(_ki67_midpoint).astype("Float64"),
        kind="continuous",
        active=True,
        overwrite=False,
        reason="Midpoint of Ki-67 range strings.",
    ),
    _c06.Apply(
        name="ki67_group",
        source="ki67_mid",
        fn=lambda s: s.map(_ki67_group),
        kind="ordinal",
        ordered_levels=["low_le_4", "intermediate_5_9", "high_ge_10"],
        active=True,
        overwrite=False,
        reason="Ki-67 clinical groups: ≤4 / 5-9 / ≥10.",
    ),
    _c06.Compute(
        name="edema_volume_cm3",
        sources=["perifocal_edema", "edema_volume_cm3"],
        fn=lambda d: d["edema_volume_cm3"].mask(
            d["perifocal_edema"].fillna(True).astype(float) == 0, 0
        ),
        kind="continuous",
        active=True,
        overwrite=True,
        reason="No perifocal edema => edema volume is structurally 0, not missing.",
    ),
    ]


In [ ]:
df, schema, derivation_log = _c06.apply_derivations(
    df=df,
    schema=schema,
    derivations=DERIVATIONS,
    output_root=OUTPUT_ROOT,
    write_csv=True,
)

derivation_log

## 10. DDA — second pass


In [ ]:
from cleaning import format_table_for_display
from IPython.display import display

dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))


## 11. Analysis targets & predictors

Edit lists, then run.

- **`INFERENTIAL_MODEL_VARIANTS`** — named multivariable calculator specs: `(id, title, link, target, [predictors])`. `link` is shown as a clickable **source** in the HTML report (use `""` if none). Each variant gets its own outcome, model, EPV bar, forest plot, VIF, and report block.


In [ ]:
#df.columns.to_list()

In [ ]:
EDA_TARGETS = ['high_grade', 'progesterone_pos', 'brain_invasion', 'ki67_group', 'hist_necrosis']
EDA_PREDICTORS = [
    'age',
    'age_bins',
    'sex',
    
    #'who_grade', ==> TARGET
    #'high_grade', ==> TARGET
    
    #'progesterone_pos',
    #'ki67_pct',
    #'ki67_mid',
    #'ki67_group'
    #'brain_invasion',
    #'hist_necrosis',
    
    'side',
    'tumor_location',
    'meningioma_count',
    'multiple_meningiomas',
    'max_diameter_cm',
    'tumor_volume',
    
    'tumor_episode',
    'tumor_margin',
    'dural_tail',
    
    'perifocal_edema',
    'edema_volume_cm3',
    
    'mass_effect',
    'calcification',
    'cystic_component',
    'necrosis',
    'hemorrhage',
    'hyperostosis',
    'cortical_destruction',
    'sinus_invasion',
    'transfalcine_extension',
    
    'capsular_enhancement',
    'heterogeneous_enhancement',
    'dwi_hyperintensity',
    't2_hyperintensity',
    't1_hypointensity',
    
    'adc_value',
    ]

INFERENTIAL_TARGETS = ['high_grade']
INFERENTIAL_PREDICTORS = [
    #'age',
    #'age_bins',

    #'sex',
    #'side',
    #'tumor_location',

    #'meningioma_count',
    'multiple_meningiomas',

    #'max_diameter_cm',
    #'tumor_volume',
    #'tumor_episode',
    #'tumor_margin',
    #'dural_tail',
    #'capsular_enhancement',
    #'heterogeneous_enhancement',

    #'perifocal_edema',
    'edema_volume_cm3',

    #'mass_effect',
    #'calcification',
    #'cystic_component',
    #'mri_necrosis',
    #'hemorrhage',
    'hyperostosis',
    #'cortical_destruction',
    #'dwi_hyperintensity',
    #'t2_hyperintensity',
    #'t1_hypointensity',
    #'sinus_invasion',
    #'transfalcine_extension',
    'adc_value',
    ]

#🟧🟧🟧 Multivariable model variants — each gets its own EPV bar, forest plot, VIF, table, interpretation.
# Use (id, title, link, target, [predictors]) or {"id": ..., "title": ..., "link": ..., "target": ..., "predictors": [...]}.
INFERENTIAL_MODEL_VARIANTS = [
    ("experimental", "meningioma_atypier experimental", "", "high_grade", INFERENTIAL_PREDICTORS),

    # Research work: Predicting the grade of meningiomas by clinical–radiological features: A comparison of precontrast and postcontrast MRI
    # Authors: Yuan Yao, Yifan Xu, Shihe Liu, Feng Xue, Bao Wang, Shanshan Qin, Xiubin Sun, Jingzhen He
    # Link: https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full
    (
        "yao_et_al_2022",
        "Yao et al. 2022 | precontrast / semantic MRI model",
        "https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full",
        "high_grade",
        [
            "sex",
            "tumor_margin",
            "cystic_component",
            "perifocal_edema",
            "dural_tail",
        ],
    ),

    # Research work: Preoperative Prediction of Intracranial Meningioma Grade Using Conventional CT and MRI
    # Authors: T. Amano et al.
    # Link: https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri
    (
        "amano_et_al_2021_expanded_proxy",
        "Amano et al. 2021 expanded proxy | conventional CT/MRI + tumor burden",
        "https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "heterogeneous_enhancement",
            "perifocal_edema",
            "tumor_volume",
            "cortical_destruction",
        ],
    ),

    # Research work: The Role of Pre-Operative MRI for Prediction of High-Grade Intracranial Meningioma: A Retrospective Study
    # Authors: Kan Radeesri, Vitit Lekhavat
    # Link: https://journal.waocp.org/article_90552.html
    (
        "radeesri_lekhavat_2020",
        "Radeesri & Lekhavat 2020 | edema / necrosis MRI model",
        "https://journal.waocp.org/article_90552.html",
        "high_grade",
        [
            "perifocal_edema",
            "edema_volume_cm3",
            "mri_necrosis",
            "hemorrhage",
            "hyperostosis",
            "mass_effect",
        ],
    ),

    # Research work: Role of ADC values and ratios of MRI scan in differentiating typical, atypical and anaplastic meningiomas
    # Authors: M. Azeemuddin et al.
    # Link: https://pubmed.ncbi.nlm.nih.gov/30317276/
    (
        "azeemuddin_et_al_2018",
        "Azeemuddin et al. 2018 | diffusion-augmented MRI model",
        "https://pubmed.ncbi.nlm.nih.gov/30317276/",
        "high_grade",
        [
            "adc_value",
            "dwi_hyperintensity",
            "tumor_location",
            "tumor_margin",
            "perifocal_edema",
            "heterogeneous_enhancement",
            "sex",
        ],
    ),

    # Research work: Diagnostic nomogram model for predicting preoperative pathological grade of meningioma
    # Authors: Shijun Peng, Zhihua Cheng, Zhilin Guo
    # Link: https://tcr.amegroups.org/article/view/55552/html
    (
        "peng_cheng_guo_2021",
        "Peng, Cheng & Guo 2021 | interface / invasion model",
        "https://tcr.amegroups.org/article/view/55552/html",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "sinus_invasion",
            "cortical_destruction",
            "mass_effect",
            "edema_volume_cm3",
            "hyperostosis",
        ],
    ),
]

In [ ]:
_c07 = load("07_analysis")
(
    EDA_TARGETS,
    EDA_PREDICTORS,
    INFERENTIAL_TARGETS,
    INFERENTIAL_PREDICTORS,
    EDA_POSITIVE_CLASS,
    INFERENTIAL_POSITIVE_CLASS,
) = _c07.resolve_analysis(
    df,
    EDA_TARGETS,
    EDA_PREDICTORS,
    INFERENTIAL_TARGETS,
    INFERENTIAL_PREDICTORS,
)
INFERENTIAL_MODEL_VARIANTS = _c07.resolve_inferential_variants(
    df, INFERENTIAL_MODEL_VARIANTS,
)

## 10. EDA — univariate screening

Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§11) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [ ]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
    )

diag_acc = screen_diagnostic_accuracy(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

#assoc[assoc['fdr_significant']]

In [ ]:
#🟧🟧🟧 Full table
#assoc

## 11. Multiple imputation (MICE)

We generate **m=10** imputed datasets via sklearn's IterativeImputer
(RandomForest estimator, separate random seed per imputation).
The pooled inferential stage applies Rubin's rules over these 10 fits.

For a quick screening run set `m=3`. For publication use `m≥10`.


In [ ]:
M = 3  # number of imputations; reduce to 3 for fast iteration
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", module=r"sklearn\.utils\.parallel")
    imputed_frames = mice_impute(df, schema, m=M, max_iter=25,
                                 random_state=42, output_root=OUTPUT_ROOT)

print(f"Generated {len(imputed_frames)} imputed frames")

print("NaN count in first imputed frame:", imputed_frames[0].isna().sum().sum())

## 12. Multivariable logistic regression (Rubin-pooled)

For each **model variant** (see `INFERENTIAL_MODEL_VARIANTS` in §11 — each row specifies its own target):

1. Build design matrix (continuous z-scored, ordinal kept as codes, nominal one-hot).
2. Iteratively drop predictors with **VIF > 5** to handle collinearity.
3. Fit logistic regression on each of the m imputed frames.
4. Pool coefficients with **Rubin's rules** (Barnard–Rubin df).
5. Report adjusted OR with 95% CI and pooled p-value.
6. Save a forest plot SVG, VIF table, calculator JSON, and report section per variant.


In [ ]:
#🟧🟧🟧 Skip MICE for now — median/mode imputation (binary left NaN by default)

#_df_pre_impute = df.copy()
#imputed_frames = [simple_impute(df, schema, impute_binary=False)]

#display(imputation_audit(
#    _df_pre_impute,
#    imputed_frames[0],
#    schema,
#    INFERENTIAL_PREDICTORS,
#    impute_binary=False,
#))

#print("NaN count (all columns):", imputed_frames[0].isna().sum().sum())

In [ ]:
display(summarize_multivariable_cases(
    imputed_frames[0],
    schema,
    targets=INFERENTIAL_TARGETS,
    variants=INFERENTIAL_MODEL_VARIANTS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    vif_threshold=5.0,
))

In [ ]:
from statsmodels.tools.sm_exceptions import ConvergenceWarning as SMConvergenceWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SMConvergenceWarning)
    inf_results = run_inferential(
        imputed_frames, schema,
        targets=INFERENTIAL_TARGETS,
        variants=INFERENTIAL_MODEL_VARIANTS,
        positive_class=INFERENTIAL_POSITIVE_CLASS,
        vif_threshold=5.0,
        output_root=OUTPUT_ROOT,
    )
#inf_results

## 12. Report (§08)

Builds `report.html` from artifacts already in `output/` (DDA, EDA, inferential).
Edit the settings cell, then run both cells.

- **`REPORT_TITLE` / `REPORT_AUTHOR`** — shown on the cover.
- **`REPORT_PATH`** — where to write the HTML file.
- **`analysis_years`** — optional cohort label suffix on the title (from §02b).


In [ ]:
REPORT_TITLE = "REPORT: Non-invasive radiological biomarkers of meningiomas as a prognostic tool for predicting tumor histological grade"
REPORT_AUTHOR = "Andris & the radio team"
REPORT_PATH = OUTPUT_ROOT / "report" / "report.html"

In [ ]:
_c08 = load("08_report_settings")
_c08.run_report(
    output_root=OUTPUT_ROOT,
    report_title=REPORT_TITLE,
    report_author=REPORT_AUTHOR,
    report_path=REPORT_PATH,
    analysis_years=ANALYSIS_YEARS,
    eda_targets=EDA_TARGETS,
)

## 13. Outputs

Everything is saved under `output/`.


In [ ]:
from pathlib import Path
for p in sorted(Path(OUTPUT_ROOT).rglob('*')):
    if p.is_file():
        print(p)